In [1]:
#Import necessary modules
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from statistics import *
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from matplotlib.backends.backend_pdf import PdfPages
import math
from scipy import stats
import sys
import copy
from collections import Counter
from collections import defaultdict
from datetime import datetime
from sklearn.ensemble import IsolationForest

In [2]:
patients=[1135, 1450, 1464, 1497, 1504, 
          1511, 1541, 1559, 1572, 1582, 
          1586, 1603, 1607, 1609, 1615, 
          1617, 1628, 1657, 1660, 1706]

In [3]:
def readData(patient):
    tm = pd.read_csv(f"{patient}_cleaned_removedays"'.csv') #solved double coverage problem
    out = np.array(tm).tolist()  

    for i in range(len(out)):
        if 'Bedroom' in out[i][1]:
            out[i][1]='Bedroom'
            
        if 'Bathroom' in out[i][1]:
            out[i][1]='Bathroom'
    
        if 'Sensor Line' in out[i][1] or 'Extra sensor line' in out[i][1]:
            out[i][1]='Hallway'
            
        if 'Hallway' in out[i][1]:
            out[i][1]= 'Hallway'
            
        if 'Walk-in Closet' in out[i][1]:
            out[i][1]= 'Walk-in Closet'
            
        if 'Kitchen' in out[i][1]:
            out[i][1]= 'Kitchen'
            
        if 'Dining Room' in out[i][1]:
            out[i][1]= 'Dining Room'
            
#         if 'Lounge' in out[i][1]:
#             out[i][1]= 'Lounge'
    
    
    return out

In [4]:
def read_left(patient):
    threshold=1800
    try:
        #file.csv is an empty csv file
        tm = pd.read_csv(f"{patient,threshold}_left_days_outing_living"'.csv')
        # Convert 'Date' to datetime format
        tm['Date'] = pd.to_datetime(tm['Date'])
        tm['Date'] =tm['Date'].dt.strftime('%d/%m/%Y')
        left_days_hours = np.array(tm).tolist() 
    
    except pd.errors.EmptyDataError:
        left_days_hours =[]
    return left_days_hours

In [5]:
def rearrange_sensor_data(data):
    from datetime import datetime
    # Convert the date-time strings to datetime objects for correct sorting
    for record in data:
        record[0] = datetime.strptime(record[0], '%d/%m/%Y %H:%M:%S')
    # Sorting the data by datetime
    sorted_data = sorted(data, key=lambda x: x[0])
    # Rearranging the inactive events


    # Convert datetime objects back to strings
    for record in sorted_data:
        record[0] = record[0].strftime('%d/%m/%Y %H:%M:%S')
    return sorted_data

In [6]:
# def readData_visitor_training(patient):
#     from datetime import datetime
#     tm = pd.read_csv(f"{patient}_Ensemblel_visitor_alldata_50th"'.csv')#_Ensemblel_visitor_alldata
    
#     visitor=np.array(tm).tolist() 
    
#     # Convert the date format in 'visitor' to match the other two lists
#     visitor = [[datetime.strptime(date_str, '%Y-%m-%d').strftime('%d/%m/%Y'), hour] for date_str, hour in visitor]

#     return visitor


In [7]:
def readData_visitor(patient):
    from datetime import datetime
    tm = pd.read_csv(f"{patient}_Ensemblel_visitor_alldata_50th"'.csv')
    
    visitor=np.array(tm).tolist() 
    
    # Convert the date format in 'visitor' to match the other two lists
    visitor = [[datetime.strptime(date_str, '%Y-%m-%d').strftime('%d/%m/%Y'), hour] for date_str, hour in visitor]

    return visitor



In [8]:
def filter_data(patient):
    from datetime import datetime, timedelta
    import pandas as pd
    # Provided data
    data = readData(patient)
    # Convert to DataFrame
    df = pd.DataFrame(data, columns=['DateTime', 'Location'])
    # Convert the DateTime column to datetime objects
    df['DateTime'] = pd.to_datetime(df['DateTime'], format='%d/%m/%Y %H:%M:%S')

    # Convert DataFrame entries back to the list of lists with DateTime formatted as string
    converted_data = [
        [row['DateTime'].strftime('%d/%m/%Y %H:%M:%S'), row['Location']]
        for index, row in df.iterrows()]
    
    return converted_data


In [9]:
def process_data(patient):
    # Read data
    out = readData(patient)
    out_df = pd.DataFrame(out, columns=['Timestamp', 'Location'])
    out_df['Timestamp'] = pd.to_datetime(out_df['Timestamp'], format='%d/%m/%Y %H:%M:%S')
    out_df['Date'] = out_df['Timestamp'].dt.date
    out_df['Hour'] = out_df['Timestamp'].dt.hour

    # Find the start date and adjust to the first day of the next month if not already the 1st
    start_date = out_df['Date'].min()
    if start_date.day != 1:
        start_date = (start_date.replace(day=1) + timedelta(days=32)).replace(day=1)
    
    # Calculate the end date for the training period (6 months after the adjusted start date)
    end_date = (start_date + pd.DateOffset(months=6)).replace(day=1) - timedelta(days=1)

    # Split the patient data into training and testing sets
    training_data = out_df[(out_df['Date'] >= start_date) & (out_df['Date'] <= end_date)]
    testing_data = out_df[out_df['Date'] > end_date]
    
    # Read visitor
    visitor = readData_visitor(patient)
    visitor_df = pd.DataFrame(visitor, columns=['Date','Hour'])
    visitor_df['Date'] = pd.to_datetime(visitor_df['Date'], format='%d/%m/%Y').dt.date
#     training and test data
#     visitor_train = readData_visitor_training(patient)
#     visitor_test = readData_visitor(patient)

#     visitor_train_df = pd.DataFrame(visitor_train, columns=['Date', 'Hour'])
#     visitor_train_df['Date'] = pd.to_datetime(visitor_train_df['Date'], format='%d/%m/%Y').dt.date

#     visitor_test_df = pd.DataFrame(visitor_test, columns=['Date', 'Hour'])
#     visitor_test_df['Date'] = pd.to_datetime(visitor_test_df['Date'], format='%d/%m/%Y').dt.date

    # Process training data
    merged_train_df = training_data.merge(visitor_df, how='left', left_on=['Date', 'Hour'], right_on=['Date', 'Hour'], indicator=True)
    filtered_train_df = merged_train_df[merged_train_df['_merge'] == 'left_only']
    filtered_train_df = filtered_train_df[['Timestamp', 'Location']]  # Keeping only the original columns

    # Process testing data
    merged_test_df = testing_data.merge(visitor_df, how='left', left_on=['Date', 'Hour'], right_on=['Date', 'Hour'], indicator=True)
    filtered_test_df = merged_test_df[merged_test_df['_merge'] == 'left_only']
    filtered_test_df = filtered_test_df[['Timestamp', 'Location']]  # Keeping only the original columns

    # Convert to lists
    training_out = np.array(filtered_train_df).tolist()
    testing_out = np.array(filtered_test_df).tolist()

    # Combine training and testing data into one list
    combined_out = training_out + testing_out

    return combined_out

In [10]:
# import calendar
# for num in patients:
#     patient=num
#     filtered_out_df= process_data(patient)
#     out = np.array(filtered_out_df).tolist() 


In [11]:
def all_missing(left,visitor):
    import datetime
    # Combine the 'left' and 'visitor' lists and convert the date strings to datetime.date objects
    missing_data_points = []
    for record in left + visitor:
        # Parse the date from the first element and the hour from the second element
        date = datetime.datetime.strptime(record[0], "%d/%m/%Y").date()
        hour = record[2]
        missing_data_points.append((date, hour))
    return missing_data_points

In [12]:
def readdate(out): #read all date
    new_date=[]
    for i in range(len(out)):
        dt = out[i][0].split(' ')[0]
        if dt not in new_date:
            new_date.append(dt)
    return new_date

### split data into three segments

In [13]:
#Nighttime---
Nighttime_1135=[0,1,2,3,4,5,6]
Nighttime_1450=[2,3,4,5]
Nighttime_1464=[0,1,2,3,4,5,6,7]
Nighttime_1497=[3,4,5,6,7,8,9] 
Nighttime_1498=[0,1,2,3,4,5]

Nighttime_1504=[3,4,5,6,7]
Nighttime_1506=[0,1,2,3,4,5,6,7]
Nighttime_1508=[0,1,2,3,4,5,6]
# Nighttime_1507=[1,2,3,4,5,6,7]
Nighttime_1511=[1,2,3,4,5,6,7]
Nighttime_1514=[0,1,2,3]

Nighttime_1522=[0,1,2,3,4,5,6]
Nighttime_1526=[1,2,3,4,5,6,7,8]
Nighttime_1540=[0,1,2,3,4,5,6,7]
Nighttime_1541=[1,2,3,4,5]
Nighttime_1559=[0,1,2,3,4,5,6]


# Nighttime_1571=[0,1,2,3,4,5]
Nighttime_1572=[0,1,2,3,4,5,6]
Nighttime_1582=[0,1,2,3,4,5]
Nighttime_1586=[0,1,2,3,4,5,6]#
Nighttime_1590=[2,3,4,5,6]
Nighttime_1592=[0,1,2,3,4,5,6]


Nighttime_1600=[0,1,2,3,4,5,6,7]
Nighttime_1603=[0,1,2,3,4,5]#
Nighttime_1607=[3,4,5,6,7,8,9]
Nighttime_1609=[1,2,3,4,5,6]#
Nighttime_1610=[0,1,2,3,4]

Nighttime_1615=[0,1,2,3,4,5,6,7,8]
Nighttime_1617=[0,1,2,3,4,5,6]
Nighttime_1628=[1,2,3,4,5,6,7,8,9,10,11,12]
Nighttime_1657=[0,1,2,3,4,5,6,7,8,9,10,11]#
Nighttime_1660=[0,1,2,3,4,5,6,7]


Nighttime_1665=[1,2,3,4,5,6,7,8,9,10]#
Nighttime_1699=[0,1,2,3,4,5,6]
Nighttime_1706=[0,1,2,3,4,5,6]#
Nighttime_1707=[0,1,2,3,4,5,6]#
Nighttime_1724=[4,5,6,7]


# Nighttime_1618=[2,3,4,5,6,7]
# Nighttime_1693=[2,3,4,5,6,7,8,9]##
# Nighttime_1627=[0,1,2,3,4,5,6,7]
# Nighttime_1542=[0,1,2,3,4,5]
# Nighttime_1591=[2,3,4,5,6,7]
# Nighttime_1754=[0,1,2,3,4,5,6,7,8,9]#

# nighttimes=[Nighttime_1135,Nighttime_1450,Nighttime_1464,Nighttime_1497,Nighttime_1498,
#             Nighttime_1504,Nighttime_1506,Nighttime_1508,Nighttime_1511,Nighttime_1514,
#             Nighttime_1522,Nighttime_1526,Nighttime_1540,Nighttime_1541,Nighttime_1559,
#             Nighttime_1572,Nighttime_1582,Nighttime_1586,Nighttime_1590,Nighttime_1592,
#             Nighttime_1600,Nighttime_1603,Nighttime_1607,Nighttime_1609,Nighttime_1610,
#             Nighttime_1615,Nighttime_1617,Nighttime_1628,Nighttime_1657,Nighttime_1660,
#             Nighttime_1665,Nighttime_1699,Nighttime_1706,Nighttime_1707,Nighttime_1724]
nighttimes=[Nighttime_1135,Nighttime_1450,Nighttime_1464,Nighttime_1497,Nighttime_1504,
            Nighttime_1506,Nighttime_1508,Nighttime_1511, Nighttime_1526,Nighttime_1541,
            Nighttime_1559, Nighttime_1572,Nighttime_1582,Nighttime_1586,Nighttime_1592,
            Nighttime_1603,Nighttime_1607,Nighttime_1609, Nighttime_1615,Nighttime_1617,
            Nighttime_1628,Nighttime_1657,Nighttime_1665,Nighttime_1706]
# Initialize an empty dictionary to hold the results
nighttimes_dict = {}

# Make a copy of the list of keys in globals() to safely iterate
global_items = list(globals().items())

# Iterate over the copied list of global items
for key, value in global_items:
    # Check if the key starts with 'Nighttime_' and the value is a list
    if key.startswith('Nighttime_') and isinstance(value, list):
        # Extract the numeric part of the key
        numeric_part = int(key.split('_')[1])
        # Add to the nighttimes_dict
        nighttimes_dict[numeric_part] = value
# nighttimes_dict
sundowning_hours_general=list(range(15,24))
# sundowning_hours_general

sundowning_hours=[]
for each_night in nighttimes:
    # Start with the general sundowning hours
    sundown = list(sundowning_hours_general)
    
    # If the nighttime list does not start with 0, add the missing hours to sundowning_hours
    if each_night[0] != 0:
        sundown.extend(range(0, each_night[0]))
        
    # Remove any hours that are in the nighttime list
    sundown = [hour for hour in sundown if hour not in each_night]
    
    # Append the updated sundown list to sundowning_hours
    sundowning_hours.append(sundown)
    
    
hours=[i for i in range(0,24)]
# Calculate daytime hours for each user by removing nighttime and sundowning hours from the total hours
daytimes = []
for i in range(len(nighttimes)):
    # Combine nighttime and sundowning hours and remove duplicates
    excluded_hours = set(nighttimes[i] + sundowning_hours[i])
    # Calculate daytime hours by excluding the combined hours from the total hours
    daytime_hours = [hour for hour in hours if hour not in excluded_hours]
    daytimes.append(daytime_hours)

# daytimes


In [14]:
def find_segmented_data(segmented_data_train):
    Datetime=[]
    Nighttime=[]
    Sundown=[]
    for hour, firings in segmented_data_train['daytime'].items():
        Datetime.extend(firings)

    for hour, firings in segmented_data_train['nighttime'].items():
        Nighttime.extend(firings)

    for hour, firings in segmented_data_train['sundowning_hours'].items():
        Sundown.extend(firings)
    return Datetime,Nighttime,Sundown

## testing

In [15]:
def split_episodes_final_corrected(df, N_seconds):
    """
    Split the DataFrame into episodes based on time gaps.
    Each episode consists of consecutive entries.
    A new episode starts only if the next entry occurs more than N_seconds after the previous entry.
    """
    episodes = []
    current_episode = []
    last_timestamp = None

    for _, row in df.iterrows():
        if last_timestamp and (row['timestamp'] - last_timestamp).total_seconds() > N_seconds:
            # If the gap to the last timestamp is more than N_seconds, start a new episode
            if current_episode:
                episodes.append(current_episode)
                current_episode = []
        # Add current row to the current episode
        current_episode.append((row['timestamp'], row['location']))
        # Update last_timestamp to the current row's timestamp
        last_timestamp = row['timestamp']

    # Add the last episode if not empty
    if current_episode:
        episodes.append(current_episode)

    return episodes



def rearrange_sensor_data(data):
    from datetime import datetime
    # Convert the date-time strings to datetime objects for correct sorting
    for record in data:
        record[0] = datetime.strptime(record[0], '%d/%m/%Y %H:%M:%S')
    # Sorting the data by datetime
    sorted_data = sorted(data, key=lambda x: x[0])
    return sorted_data



In [16]:
def select_data(training_date):
    if len(training_date)==0:
        return ("no data")
    
    else:
        filtered_Direct_list = [pair for pair in Direct_list if pair[0][0].date() in training_date]
        filtered_Pacing_list = [pair for pair in Pacing_list if pair[0][0].date() in training_date]
        filtered_Lapping_list = [pair for pair in Lapping_list if pair[0][0].date() in training_date]
        filtered_Random_list = [pair for pair in Random_list if pair[0][0].date() in training_date]

#     #     print(len(filtered_Direct_list),len(filtered_Pacing_list))
#         inefficient_ratio=round(100-(len(filtered_Direct_list)/(
#             len(filtered_Direct_list)+len(filtered_Pacing_list)+len(filtered_Lapping_list)+len(filtered_Random_list))*100),2)

    return filtered_Direct_list,filtered_Pacing_list,filtered_Lapping_list,filtered_Random_list


In [17]:
from collections import defaultdict

def segmentation_patterns(num, filtered_Direct_list):
    for m in range(len(patients)):
        if num == patients[m]:
            Daytime = daytimes[m]
            Nighttime = nighttimes[m]
            Sundown = sundowning_hours[m]

            segmented_data = {
                "sundowning_hours": defaultdict(list),
                "nighttime": defaultdict(list),
                "daytime": defaultdict(list)
            }

            for pair in filtered_Direct_list:
                start_timestamp = pair[0][0]  
                hour_of_day = start_timestamp.hour

                if hour_of_day in Sundown:
                    segmented_data["sundowning_hours"][start_timestamp.date()].append(pair)
                elif hour_of_day in Nighttime:
                    segmented_data["nighttime"][start_timestamp.date()].append(pair)
                elif hour_of_day in Daytime:
                    segmented_data["daytime"][start_timestamp.date()].append(pair)

            return segmented_data

    return None  # Return None or an empty dict if no patient matches


In [18]:
def check_and_simplify_cross_occurrence(locations):
    if len(locations) < 4:
        return locations  # 如果列表太短，无法形成交叉模式，直接返回
    
    # 查找并简化连续交叉模式
    i = 0
    while i < len(locations) - 3:
        pattern = [locations[i], locations[i+1]]
         # check if the pattern in visible_rooms_converted
        if (locations[i], locations[i+1]) in visible_rooms_converted:
            
            # 检测连续交叉模式，如 [5, 6, 5, 6]
            if locations[i] == locations[i+2] and locations[i+1] == locations[i+3]:
                # 向后检查此模式是否继续
                j = i + 4
                while j < len(locations) and locations[j:j+2] == pattern:
                    j += 2
                # 移除除了模式的第一次出现之外的所有重复
                locations = locations[:i+2] + locations[j:]
                i += 2  # 继续检查列表的下一部分
            else:
                i += 1
        else:
            break
    return locations

In [19]:
def inefficient_ration_cal(string):
    efficient_ratio= False
    if string=='daytime':
        lengths_Direct= len([list_of_tuples for lists_of_tuples in segmented_data_Direct['daytime'].values() for list_of_tuples in lists_of_tuples])
        lengths_Pacing= len([list_of_tuples for lists_of_tuples in segmented_data_Pacing['daytime'].values() for list_of_tuples in lists_of_tuples])
        lengths_Lapping= len([list_of_tuples for lists_of_tuples in segmented_data_Lapping['daytime'].values() for list_of_tuples in lists_of_tuples])
        lengths_Random = len([list_of_tuples for lists_of_tuples in segmented_data_Random['daytime'].values() for list_of_tuples in lists_of_tuples])



    elif string=='nighttime':
        lengths_Direct= len([list_of_tuples for lists_of_tuples in segmented_data_Direct['nighttime'].values() for list_of_tuples in lists_of_tuples])
        lengths_Pacing= len([list_of_tuples for lists_of_tuples in segmented_data_Pacing['nighttime'].values() for list_of_tuples in lists_of_tuples])
        lengths_Lapping= len([list_of_tuples for lists_of_tuples in segmented_data_Lapping['nighttime'].values() for list_of_tuples in lists_of_tuples])
        lengths_Random = len([list_of_tuples for lists_of_tuples in segmented_data_Random['nighttime'].values() for list_of_tuples in lists_of_tuples])


    else:
        lengths_Direct= len([list_of_tuples for lists_of_tuples in segmented_data_Direct['sundowning_hours'].values() for list_of_tuples in lists_of_tuples])
        lengths_Pacing= len([list_of_tuples for lists_of_tuples in segmented_data_Pacing['sundowning_hours'].values() for list_of_tuples in lists_of_tuples])
        lengths_Lapping= len([list_of_tuples for lists_of_tuples in segmented_data_Lapping['sundowning_hours'].values() for list_of_tuples in lists_of_tuples])
        lengths_Random = len([list_of_tuples for lists_of_tuples in segmented_data_Random['sundowning_hours'].values() for list_of_tuples in lists_of_tuples])

    if (lengths_Direct+lengths_Pacing+lengths_Lapping+lengths_Random)>0:
            
        efficient_ratio=lengths_Direct/(lengths_Direct+lengths_Pacing+lengths_Lapping+lengths_Random)

 
            
    print('Direct',lengths_Direct,
          'Pacing',lengths_Pacing,
          'Lapping',lengths_Lapping,
          'Random',lengths_Random)
    
    if efficient_ratio!= False:
        inefficient_ratio =round((1-efficient_ratio)*100,2)
    else:
        inefficient_ratio= 0
        
    return inefficient_ratio

In [20]:
def select_data_test(training_date):
    from pandas import Timestamp
    
    filtered_Direct_list = [pair for pair in Direct_list if pair[0][0].date() == training_date]
    filtered_Pacing_list = [pair for pair in Pacing_list if pair[0][0].date() == training_date]
    filtered_Lapping_list = [pair for pair in Lapping_list if pair[0][0].date() == training_date]
    filtered_Random_list = [pair for pair in Random_list if pair[0][0].date() == training_date]

    return filtered_Direct_list,filtered_Pacing_list,filtered_Lapping_list,filtered_Random_list


In [21]:
from datetime import datetime, timedelta

def generate_monthly_date_ranges(start_date, total_segments=None):
    current_start_date = start_date
    segments_generated = 0
    
    while total_segments is None or segments_generated < total_segments:
        # Calculate the last day of the current month
        next_month = current_start_date.month % 12 + 1
        next_month_year = current_start_date.year + (current_start_date.month // 12)
        # Set day to the first of the next month, then subtract one day to get the end of the current month
        current_end_date = (datetime(next_month_year, next_month, 1) - timedelta(days=1)).date()
        
        yield (current_start_date, current_end_date)
        
        # Update the start date to the first of the next month
        current_start_date = current_end_date + timedelta(days=1)
        segments_generated += 1


In [22]:
# Function to check for missing dates within each chunk
def find_missing_dates(date_ranges, cleaned_data_all):
    # Convert cleaned_data dates to datetime.date objects
    cleaned_dates = [entry[0].date() for entry in cleaned_data_all]

    missing_dates_by_range = []
    for start_date, end_date in date_ranges:
        expected_dates = {start_date + timedelta(days=x) for x in range((end_date - start_date).days + 1)}
        missing_dates = expected_dates - set(cleaned_dates)
        missing_dates_by_range.append((start_date, end_date, sorted(list(missing_dates))))
    return missing_dates_by_range



In [23]:
# Function to replace specific room names with a generalized category
def replace_room_name(room):
    if 'Bathroom' in room:
        return 'Bathroom'
    elif 'Bedroom' in room :
        return 'Bedroom'
    elif 'Living Room' in room:
        return 'Living Room'
    
    elif 'Sensor Line' in room or 'Extra sensor line' in room:
        return 'Hallway'
    elif 'Hallway' in room:
        return 'Hallway'
    else:
        return room


In [24]:
def read_adjacent(patient):
    tm = pd.read_csv(f"{patient}_adjacent_list"'.csv').dropna() #drop 'NaN'
    adjacent = np.array(tm).tolist()  
   
    # Apply replacement and convert to bidirectional tuples
    bidirectional_adjacency = []
    for pair in adjacent:
        # Replace room names in each pair
        updated_pair = [replace_room_name(room) for room in pair]
        # Add both directional pairs
        bidirectional_adjacency.append(tuple(updated_pair))
        bidirectional_adjacency.append(tuple(updated_pair[::-1]))
    bidirectional_adjacency=list(set(bidirectional_adjacency))
    return bidirectional_adjacency

In [25]:
def classify_episode_pattern(episode_patterns):
    Direct_list=[]
    Lapping_list=[]
    Pacing_list=[]
    Random_list=[]
    
    """
    Strict classification function for episodes, focusing on accurately identifying 'Lapping' pattern.
    """
    
    for episode in episode_patterns:
        add_list=False
        # Extract locations from the episode
        locations = [loc for _, loc, in episode if loc is not None]

        # Remove consecutive duplicates to simplify the locations list
        new_locations = [locations[0]]
        for loc in locations[1:]:
            if loc != new_locations[-1]:
                new_locations.append(loc)
#         
        
        simplified_locations=check_and_simplify_cross_occurrence(new_locations)
#         if new_locations!=simplified_locations:
#         print('new_locations',new_locations,simplified_locations)
    
    
        # Check for Lapping pattern
        # A lapping pattern requires a circular path with at least three different locations and some repetition
        if len(set(simplified_locations)) >= 3:
            # Check for repeating pattern
            half_index = len(simplified_locations) // 2
            if (simplified_locations[:half_index] == simplified_locations[half_index:]) or (simplified_locations[:half_index] == simplified_locations[:half_index-1:-1]):
#                 print(new_locations,check_and_simplify_cross_occurrence(new_locations))
#                 print('moved_locations',simplified_locations,"Lapping")
                Lapping_list.append(episode)
                add_list=True

    #             return "Lapping"

        # Check for Direct and Pacing patterns using the same logic as before
        if (len(simplified_locations) == len(set(simplified_locations))) or (len(simplified_locations)<=3):#
            Direct_list.append(episode)
#             print('moved_locations',simplified_locations,"Direct")
            add_list=True

            
        if len(simplified_locations) == 4:
            first_loc = simplified_locations[0]
            second_loc = None
            for loc in simplified_locations[1:]:
                if loc != first_loc:
                    second_loc = loc
                    break
            if second_loc and simplified_locations[:4] == [first_loc, second_loc] * 2:
                Pacing_list.append(episode)
    #             return "Pacing"
#                 print(new_locations,check_and_simplify_cross_occurrence(new_locations))
#                 print('moved_locations',simplified_locations,"Pacing")
                add_list=True

        if not add_list: #and len(simplified_locations)>=5:
            # If none of the above, it's a Random pattern
            Random_list.append(episode)
#             print(new_locations,check_and_simplify_cross_occurrence(new_locations))
#             print('moved_locations',simplified_locations,"Random")
#     return "Random"
    return Direct_list,Pacing_list,Lapping_list,Random_list



In [26]:
from datetime import datetime, timedelta

def generate_weekly_date_ranges(start_date, total_segments=None):
    current_start_date = start_date
    segments_generated = 0
    
    while total_segments is None or segments_generated < total_segments:
        # Calculate the end date of the current week
        current_end_date = current_start_date + timedelta(days=6)
        
        yield (current_start_date, current_end_date)
        
        # Update the start date to the first day of the next week
        current_start_date = current_end_date + timedelta(days=1)
        segments_generated += 1


In [ ]:
final_results={}
final_results_week={}

import calendar
for num in patients:
    patient=num
    final_results[num]={}
    final_results_week[num]={}
    filtered_out_df= process_data(patient)
    out = np.array(filtered_out_df).tolist() 


    
    
    import pandas as pd
    from datetime import datetime, timedelta
    # maaping room name to numbers
    locations = [d[1] for d in readData(patient)]
    odd_numbers = (range(1, len(set(locations))+1 )) # generate a list of odd numbers
    location_mapping = dict(zip(set(locations), odd_numbers)) 
    print(patient,location_mapping)
    
    visible_rooms=read_adjacent(num)
    
    
    visible_rooms_converted = []
    for pair in visible_rooms:
        # Check if both locations are in the mapping
        if pair[0] in location_mapping and pair[1] in location_mapping:
            visible_rooms_converted.append((location_mapping[pair[0]], location_mapping[pair[1]]))
    
    
#     visible_rooms_converted = [(location_mapping[pair[0]], location_mapping[pair[1]]) for pair in visible_rooms]
    
    
    # find episodes
    data = out
    df = pd.DataFrame(data, columns=['timestamp', 'location'])
    df['location'] = df['location'].map(location_mapping)
    df['timestamp'] = pd.to_datetime(df['timestamp'], format='%d/%m/%Y %H:%M:%S')
    # episode_patterns
    episodes = split_episodes_final_corrected(df, N_seconds=10)
    Direct_list,Pacing_list,Lapping_list,Random_list=classify_episode_pattern(episodes)

    
    unique_dates = df['timestamp'].dt.date.unique()
    # Sort the dates to find the earliest dates
    sorted_dates = sorted(unique_dates)
    dates=sorted_dates

   
    from datetime import datetime, timedelta
    dates=sorted_dates
    start_date = dates[0]
    end_date = dates[-1]

    # date_ranges
    
    total_days = (end_date - start_date).days + 1
    
    
#(1) monthly
    total_segments = (total_days + 29) // 30  
    date_ranges = list(generate_monthly_date_ranges(start_date, total_segments=total_segments))
    # # Find missing dates within each date range
    missing_dates_by_range = find_missing_dates(date_ranges, out)
    
    
    #testing
    for start_date, end_date in date_ranges:
        expected_dates = {start_date + timedelta(days=x) for x in range((end_date - start_date).days + 1)}
        for start_date_missing, end_date_missing, missing_dates in missing_dates_by_range:
            if start_date_missing==start_date and start_date.day == 1:
                _, num_days_in_month = calendar.monthrange(start_date.year, start_date.month)
                print(f"From {start_date} to {end_date}, missing dates: {len(missing_dates)}",'num_days_in_month',num_days_in_month)

#                 missing_threshold_days = math.floor(max_threshold_days[num]*num_days_in_month)
    #             print('missing_threshold_days',max_threshold_days[num],missing_threshold_days)

                filtered_Direct_list,filtered_Pacing_list,filtered_Lapping_list,filtered_Random_list=select_data(expected_dates)
                if (len(filtered_Direct_list)+len(filtered_Pacing_list)+len(
                    filtered_Lapping_list)+len(filtered_Random_list))>0:

                    inefficient_ratio_total=round((1-(len(filtered_Direct_list)/(len(filtered_Direct_list)+
                                                                          len(filtered_Pacing_list)+
                                                                          len(filtered_Lapping_list)+
                                                                          len(filtered_Random_list))))*100,2)


                    print('Inefficient ratio in entire data in month:', start_date.month, inefficient_ratio_total,'%')

                    segmented_data_Direct=segmentation_patterns(num,filtered_Direct_list)
                    segmented_data_Pacing=segmentation_patterns(num,filtered_Pacing_list)
                    segmented_data_Lapping=segmentation_patterns(num,filtered_Lapping_list)
                    segmented_data_Random=segmentation_patterns(num,filtered_Random_list)
                    #Daytime
                    inefficient_ratio_Daytime=inefficient_ration_cal('daytime')

                    #Nighttime
                    inefficient_ratio_Nighttime=inefficient_ration_cal('nighttime')

                    #Sundowning
                    inefficient_ratio_Sundowning=inefficient_ration_cal('sundowning_hours')

                    print('daytime',inefficient_ratio_Daytime,'%',
                          'nighttime',inefficient_ratio_Nighttime,'%',
                          'sundowning',inefficient_ratio_Sundowning,'%') 
                    print('\n')
                    final_results[num][start_date]=[inefficient_ratio_total,inefficient_ratio_Daytime,
                                                   inefficient_ratio_Nighttime,inefficient_ratio_Sundowning]
                else:
                    final_results[num][start_date]=None


1135 {'Front Door': 1, 'Living Room': 2, 'Dining Room': 3, 'Bathroom': 4, 'Other 1': 5, 'Back Door': 6, 'Bedroom': 7, 'Kitchen': 8, 'Hallway': 9}
From 2018-07-01 to 2018-07-31, missing dates: 0 num_days_in_month 31
Inefficient ratio in entire data in month: 7 26.29 %
Direct 2351 Pacing 72 Lapping 29 Random 866
Direct 589 Pacing 0 Lapping 11 Random 146
Direct 2898 Pacing 87 Lapping 28 Random 843
daytime 29.14 % nighttime 21.05 % sundowning 24.84 %


From 2018-08-01 to 2018-08-31, missing dates: 0 num_days_in_month 31
Inefficient ratio in entire data in month: 8 25.13 %
Direct 3104 Pacing 95 Lapping 36 Random 1107
Direct 713 Pacing 0 Lapping 41 Random 153
Direct 3880 Pacing 74 Lapping 30 Random 1047
daytime 28.51 % nighttime 21.39 % sundowning 22.88 %


From 2018-09-01 to 2018-09-30, missing dates: 0 num_days_in_month 30
Inefficient ratio in entire data in month: 9 24.78 %
Direct 2438 Pacing 69 Lapping 28 Random 842
Direct 703 Pacing 3 Lapping 49 Random 117
Direct 4116 Pacing 97 Lapping 

Inefficient ratio in entire data in month: 1 7.98 %
Direct 9378 Pacing 33 Lapping 11 Random 866
Direct 1651 Pacing 0 Lapping 1 Random 63
Direct 9461 Pacing 48 Lapping 8 Random 748
daytime 8.85 % nighttime 3.73 % sundowning 7.83 %


From 2019-02-01 to 2019-02-28, missing dates: 0 num_days_in_month 28
Inefficient ratio in entire data in month: 2 7.42 %
Direct 9149 Pacing 28 Lapping 10 Random 789
Direct 1477 Pacing 0 Lapping 0 Random 56
Direct 8811 Pacing 54 Lapping 7 Random 613
daytime 8.29 % nighttime 3.65 % sundowning 7.11 %


From 2019-03-01 to 2019-03-31, missing dates: 0 num_days_in_month 31
Inefficient ratio in entire data in month: 3 8.43 %
Direct 9534 Pacing 20 Lapping 18 Random 959
Direct 1429 Pacing 0 Lapping 0 Random 55
Direct 9068 Pacing 62 Lapping 6 Random 723
daytime 9.47 % nighttime 3.71 % sundowning 8.02 %


From 2019-04-01 to 2019-04-30, missing dates: 0 num_days_in_month 30
Inefficient ratio in entire data in month: 4 8.31 %
Direct 7385 Pacing 19 Lapping 16 Random 722
D

Inefficient ratio in entire data in month: 7 5.89 %
Direct 1231 Pacing 0 Lapping 0 Random 118
Direct 1183 Pacing 0 Lapping 0 Random 43
Direct 2088 Pacing 1 Lapping 2 Random 118
daytime 8.75 % nighttime 3.51 % sundowning 5.48 %


From 2019-08-01 to 2019-08-31, missing dates: 7 num_days_in_month 31
Inefficient ratio in entire data in month: 8 9.91 %
Direct 1554 Pacing 6 Lapping 3 Random 175
Direct 1671 Pacing 1 Lapping 2 Random 88
Direct 3496 Pacing 11 Lapping 5 Random 448
daytime 10.59 % nighttime 5.16 % sundowning 11.72 %


From 2019-09-01 to 2019-09-30, missing dates: 0 num_days_in_month 30
Inefficient ratio in entire data in month: 9 11.44 %
Direct 1841 Pacing 8 Lapping 5 Random 275
Direct 1912 Pacing 0 Lapping 1 Random 135
Direct 3155 Pacing 19 Lapping 5 Random 444
daytime 13.53 % nighttime 6.64 % sundowning 12.92 %


From 2019-10-01 to 2019-10-31, missing dates: 0 num_days_in_month 31
Inefficient ratio in entire data in month: 10 8.15 %
Direct 1462 Pacing 5 Lapping 6 Random 231
Dir

1504 {'Front Door': 1, 'Living Room': 2, 'Office 1': 3, 'Bathroom': 4, 'Other 1': 5, 'Lounge 1': 6, 'Laundry Room 1': 7, 'Back Door': 8, 'Bedroom': 9, 'Kitchen': 10, 'Hallway': 11}
From 2018-08-01 to 2018-08-31, missing dates: 8 num_days_in_month 31
Inefficient ratio in entire data in month: 8 5.2 %
Direct 2353 Pacing 27 Lapping 1 Random 119
Direct 1580 Pacing 4 Lapping 0 Random 26
Direct 5895 Pacing 50 Lapping 4 Random 308
daytime 5.88 % nighttime 1.86 % sundowning 5.79 %


From 2018-09-01 to 2018-09-30, missing dates: 0 num_days_in_month 30
Inefficient ratio in entire data in month: 9 3.47 %
Direct 2966 Pacing 8 Lapping 1 Random 83
Direct 2188 Pacing 0 Lapping 1 Random 28
Direct 7161 Pacing 18 Lapping 7 Random 297
daytime 3.01 % nighttime 1.31 % sundowning 4.3 %


From 2018-10-01 to 2018-10-31, missing dates: 0 num_days_in_month 31
Inefficient ratio in entire data in month: 10 2.09 %
Direct 3305 Pacing 4 Lapping 0 Random 50
Direct 2653 Pacing 0 Lapping 0 Random 8
Direct 7003 Pacing 5

Inefficient ratio in entire data in month: 6 4.02 %
Direct 2308 Pacing 4 Lapping 8 Random 111
Direct 2313 Pacing 7 Lapping 7 Random 52
Direct 4899 Pacing 17 Lapping 8 Random 185
daytime 5.06 % nighttime 2.77 % sundowning 4.11 %


From 2019-07-01 to 2019-07-31, missing dates: 0 num_days_in_month 31
Inefficient ratio in entire data in month: 7 4.16 %
Direct 3121 Pacing 11 Lapping 17 Random 140
Direct 2669 Pacing 8 Lapping 4 Random 58
Direct 5473 Pacing 29 Lapping 11 Random 211
daytime 5.11 % nighttime 2.56 % sundowning 4.39 %


From 2019-08-01 to 2019-08-31, missing dates: 0 num_days_in_month 31
Inefficient ratio in entire data in month: 8 4.48 %
Direct 2871 Pacing 16 Lapping 7 Random 129
Direct 2442 Pacing 17 Lapping 5 Random 60
Direct 5030 Pacing 24 Lapping 5 Random 222
daytime 5.03 % nighttime 3.25 % sundowning 4.75 %


From 2019-09-01 to 2019-09-30, missing dates: 0 num_days_in_month 30
Inefficient ratio in entire data in month: 9 3.85 %
Direct 3410 Pacing 17 Lapping 7 Random 137
Dir

In [ ]:
Lapping_list

In [ ]:
xx

In [ ]:
# save to a csv

In [ ]:
# Dictionary comprehension to filter out None values
final_results = {
    num: {date: value for date, value in dates.items() if value is not None}
    for num, dates in final_results.items()
}


In [ ]:
# Iterate through each patient and save their data to individual CSV files
file_paths = []

for patient_id, dates in final_results.items():
    # Prepare data for this specific patient
    patient_data = []
    for date, ratios in dates.items():
        patient_data.append({
            'Month': date,
            'Inefficient_Ratio_Total': ratios[0],
            'Inefficient_Ratio_Daytime': ratios[1],
            'Inefficient_Ratio_Nighttime': ratios[2],
            'Inefficient_Ratio_Sundowning': ratios[3]
        })
    
    # Create DataFrame for the patient
    patient_df = pd.DataFrame(patient_data)
    
    # Define file path for this patient
    patient_file_path = f'{patient_id}_wandering_patterns_ours_50th.csv' #_wandering_patterns_ours_90th
    file_paths.append(patient_file_path)
    
    # Save the DataFrame to CSV
    patient_df.to_csv(patient_file_path, index=False)

# Display all file paths created
file_paths


In [ ]:
# Plotting each user's data
for user, data in final_results.items():
    # Sort the dictionary by date to ensure the plot is ordered
    sorted_dates = sorted(data.keys())
    # Extract the ratios for each date
    total = [data[date][0]/100 for date in sorted_dates]
    daytime = [data[date][1]/100 for date in sorted_dates]
    nighttime = [data[date][2]/100 for date in sorted_dates]
    sundowning = [data[date][3]/100 for date in sorted_dates]

    # Create a figure with 4 subplots (one for each ratio category)
    fig, axs = plt.subplots(4, 1, figsize=(10, 12), sharex=True)
#     fig.suptitle(f'Inefficient Ratios Over Time for User {user}')

    # Plotting
    axs[0].plot(sorted_dates, total, marker='o', linestyle='-')
    axs[0].set_title(f'Total Inefficient Ratio for User {user}')
    axs[0].set_ylabel('Ratio')
    axs[0].set_ylim([0,0.45])

    axs[1].plot(sorted_dates, daytime, marker='o', linestyle='-', color='orange')
    axs[1].set_title(f'Daytime Inefficient Ratio for User {user}')
    axs[1].set_ylabel('Ratio')
    axs[1].set_ylim([0,0.45])

    axs[2].plot(sorted_dates, nighttime, marker='o', linestyle='-', color='green')
    axs[2].set_title(f'Nighttime Inefficient Ratio for User {user}')
    axs[2].set_ylabel('Ratio')
    axs[2].set_ylim([0,0.45])

    axs[3].plot(sorted_dates, sundowning, marker='o', linestyle='-', color='red')
    axs[3].set_title(f'Sundowning Inefficient Ratio for User {user}')
    axs[3].set_ylabel('Ratio')
    axs[3].set_xlabel('Date')
    axs[3].set_ylim([0,0.45])

    # Format the date display
    plt.gcf().autofmt_xdate()

    plt.show()

In [ ]:
x

In [ ]:
x